In [18]:
import selenium
import csv

In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(options=options)
driver.get("https://www.google.com")
print("✅ Funcionando:", driver.title)
driver.quit()

✅ Funcionando: Google


In [21]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup

# Classes
BASE_URL = "https://www.tripadvisor.com.br"
USER = "QIHsu Zb"
RATING = "MyMKp u Q1"
COMMENT = "JguWG"
COMMENT_TITLE = "biGQs _P SewaP qWPrE ncFvv ezezH"
LOCAL = "biGQs P navcl"
DATE = "BNelO"


def get_driver_connection():
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--profile-directory=Default")
    options.add_argument("--disable-plugins-discovery")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {"source": """
        Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
        Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3]});
        Object.defineProperty(navigator, 'languages', {get: () => ['pt-BR', 'pt', 'en-US']});
        window.chrome = { runtime: {} };
    """})
    return driver


def add_into_csv(*dados):
    with (open("dados_csv.csv", "a", newline="") as arquivo):
        csv.writer(arquivo
                   ).writer.writerow(dados)

def extract_reviews(soup: BeautifulSoup) -> list:
    reviews = []

    card_wrapper = soup.find(class_="mSOQy emJoM")
    if not card_wrapper:
        print("  ⚠️  Wrapper de cards não encontrado.")
        return reviews

    cards = card_wrapper.find_all(class_="_c")

    for card in cards:
        try:

            user = card.find(class_=USER).a.text.strip()
            rating = card.find(class_=RATING).title.text.split()[0]
            comment = card.find(class_=COMMENT).text.strip()
            comment_title = card.find(class_=COMMENT_TITLE).text.strip()

            local = card.find(class_=LOCAL).span.text
            if local[0] >= "0" and local[0] <= "9":
                local = None

            date = card.find(class_=DATE).div.text[9:]

            # Insert into csv
            add_into_csv(user, rating, comment, comment_title, local, date)
            # reviews.append({
            #     "user": user,
            #     "rating": rating,
            #     "comment_title": comment_title,
            #     "comment": comment,
            #     "local": local,
            #     "date": date
            # })

        except Exception as e:
            print(f"  ⚠️  Erro ao extrair card: {e}")
            continue

    return reviews


def get_next_url(soup: BeautifulSoup) -> str | None:
    proxima = soup.find("a", {"data-smoke-attr": "pagination-next-arrow"})
    if proxima and proxima.get("href"):
        return proxima["href"]
    return None


# Acessa o Google antes para simular comportamento humano
driver = get_driver_connection()
driver.get("https://www.google.com.br")
time.sleep(2)

url_atual = (
    "/Attraction_Review-g680306-d2401618"
    "-Reviews-Praia_das_Laranjeiras-Balneario_Camboriu_State_of_Santa_Catarina.html"
)

todas_reviews = []
pagina = 1

count = 1
while url_atual or count == 3:
    print(f"\n📄 Página {pagina}: {BASE_URL + url_atual}")
    driver.get(BASE_URL + url_atual)
    time.sleep(4)

    driver.execute_script("window.scrollTo(0, 300);")
    time.sleep(2)
    driver.execute_script("window.scrollTo(0, 600);")
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    reviews = extract_reviews(soup)
    todas_reviews.extend(reviews)
    count+= count +1
    print(f"  ✅ {len(reviews)} reviews extraídas | Total acumulado: {len(todas_reviews)}")

    url_atual = get_next_url(soup)
    if not url_atual:
        print("\n🔚 Última página atingida.")

    pagina += 1

driver.quit()

print(f"\n📊 Total geral de reviews: {len(todas_reviews)}")
for r in todas_reviews:
    print(r)


📄 Página 1: https://www.tripadvisor.com.br/Attraction_Review-g680306-d2401618-Reviews-Praia_das_Laranjeiras-Balneario_Camboriu_State_of_Santa_Catarina.html
  ⚠️  Wrapper de cards não encontrado.
  ✅ 0 reviews extraídas | Total acumulado: 0

🔚 Última página atingida.


TypeError: can only concatenate str (not "NoneType") to str